# MetricGAN-LwF Visualisierung

Dieses Notebook visualisiert Trainingsmetriken aus mehreren Runs.

Enthalten sind:
- Ein Metrik-Plot pro Run (alle gefundenen Metriken)
- Ein gemeinsamer PESQ-Vergleichsplot über alle Runs

In [ ]:
from pathlib import Path
import re
import warnings

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Kandidaten für den Basisordner mit Ablationen
BASE_RESULTS_DIR_CANDIDATES = [
    Path("results/ablations_core"),
    Path("metricgan-lwf/results/ablations_core"),
    Path("/workspaces/MetricGAN-KAN-LwF/metricgan-lwf/results/ablations_core"),
]

# Fallback-Wurzeln für rekursive Suche
RECURSIVE_SEARCH_ROOTS = [
    Path("."),
    Path("/workspaces/MetricGAN-KAN-LwF"),
]

# Optional: explizite Run-Ordner oder train_log.txt-Pfade angeben
# Beispiele:
# - "results/ablations_core/baseline_lwf_off/4234"
# - "results/ablations_core/baseline_lwf_off/4234/train_log.txt"
MANUAL_RUN_DIRS = []

# Dateiname des SpeechBrain-Loggers
TRAIN_LOG_NAME = "train_log.txt"

In [ ]:
def discover_runs(base_dir_candidates, manual_dirs=None, log_name="train_log.txt", recursive_roots=None):
    if manual_dirs is None:
        manual_dirs = []
    if recursive_roots is None:
        recursive_roots = []

    run_dirs = []

    # 1) Manuelle Pfade priorisieren
    for p in manual_dirs:
        rp = Path(p)
        if rp.is_file() and rp.name == log_name:
            run_dirs.append(rp.parent)
        elif rp.is_dir() and (rp / log_name).exists():
            run_dirs.append(rp)
        else:
            warnings.warn(f"Kein {log_name} im manuellen Pfad: {rp}")

    # 2) Direkte Suche in bekannten Basisordnern
    existing_bases = [Path(b) for b in base_dir_candidates if Path(b).exists()]
    if not run_dirs:
        for base_dir in existing_bases:
            for log_path in sorted(base_dir.glob(f"*/*/{log_name}")):
                run_dirs.append(log_path.parent)

    # 3) Fallback: rekursiv im Workspace suchen
    if not run_dirs:
        pattern = f"**/results/ablations_core/*/*/{log_name}"
        for root in recursive_roots:
            root = Path(root)
            if not root.exists():
                continue
            for log_path in sorted(root.glob(pattern)):
                run_dirs.append(log_path.parent)

    # Duplikate entfernen, Reihenfolge beibehalten
    unique = []
    seen = set()
    for rd in run_dirs:
        r = rd.resolve()
        if r not in seen:
            seen.add(r)
            unique.append(rd)

    return unique, existing_bases


def run_label_from_dir(run_dir: Path):
    # Extrahiert robust: ablation/seed aus .../results/ablations_core/<ablation>/<seed>
    parts = run_dir.resolve().parts
    if "ablations_core" in parts:
        idx = parts.index("ablations_core")
        if idx + 2 < len(parts):
            return f"{parts[idx + 1]}/{parts[idx + 2]}"
    return run_dir.name


def parse_train_log(log_path: Path):
    text = log_path.read_text(encoding="utf-8", errors="ignore")

    # Exaktes Trainingsformat:
    # Epoch: 12 - train loss: 3.76e-01 - valid SI-SNR: 6.25, valid pesq: 1.59, valid stoi: 8.11e-01
    pattern = re.compile(
        r"^\s*Epoch:\s*(?P<epoch>\d+)\s*-\s*"
        r"train\s+loss:\s*(?P<loss>-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)\s*-\s*"
        r"valid\s+SI-SNR:\s*(?P<si_snr>-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)\s*,\s*"
        r"valid\s+pesq:\s*(?P<pesq>-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)\s*,\s*"
        r"valid\s+stoi:\s*(?P<stoi>-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)\s*$",
        flags=re.IGNORECASE,
    )

    rows = []
    for raw_line in text.splitlines():
        m = pattern.match(raw_line.strip())
        if not m:
            continue
        rows.append(
            {
                "epoch": float(m.group("epoch")),
                "loss": float(m.group("loss")),
                "si_snr": float(m.group("si_snr")),
                "pesq": float(m.group("pesq")),
                "stoi": float(m.group("stoi")),
            }
        )

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows).sort_values("epoch")
    df = df.drop_duplicates(subset=["epoch"], keep="last").reset_index(drop=True)
    return df

In [ ]:
run_dirs, existing_bases = discover_runs(
    BASE_RESULTS_DIR_CANDIDATES,
    MANUAL_RUN_DIRS,
    TRAIN_LOG_NAME,
    RECURSIVE_SEARCH_ROOTS,
)

print(f"Arbeitsverzeichnis: {Path.cwd()}")
print("Gefundene existierende Basisordner:")
for b in existing_bases:
    print(f"- {b}")

if not run_dirs:
    print("Keine Runs gefunden. Prüfe BASE_RESULTS_DIR_CANDIDATES oder setze MANUAL_RUN_DIRS.")
else:
    print("Gefundene Runs:")
    for r in run_dirs:
        print(f"- {r}")

In [ ]:
run_data = {}

for run_dir in run_dirs:
    log_path = run_dir / TRAIN_LOG_NAME
    df = parse_train_log(log_path)

    if df.empty:
        warnings.warn(f"Keine Trainingsmetriken gefunden in {log_path}")
        continue

    run_label = run_label_from_dir(run_dir)
    run_data[run_label] = df

print(f"Eingelesene Runs mit Metriken: {len(run_data)}")
for run_name, df in run_data.items():
    print(f"- {run_name}: Epochen={len(df)}, PESQ max={df['pesq'].max():.3f}")

In [ ]:
# 1) Pro Run: Metrikentwicklung
for run_name, df in run_data.items():
    metric_cols = [c for c in df.columns if c != "epoch"]
    if not metric_cols:
        continue

    fig, ax = plt.subplots(figsize=(11, 5))
    for col in metric_cols:
        ax.plot(df["epoch"], df[col], marker="o", linewidth=1.8, markersize=3, label=col)

    ax.set_title(f"Run: {run_name} | Metrikentwicklung", fontsize=13)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Wert")
    ax.legend(loc="best", ncols=min(4, max(1, len(metric_cols))))
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
# 2) Gemeinsamer Plot: alle PESQ-Kurven
fig, ax = plt.subplots(figsize=(11, 5))
found_pesq = False

for run_name, df in run_data.items():
    if "pesq" not in df.columns:
        continue
    found_pesq = True
    ax.plot(df["epoch"], df["pesq"], marker="o", linewidth=2.0, markersize=3, label=run_name)

if found_pesq:
    ax.set_title("PESQ-Vergleich über alle Läufe", fontsize=13)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("PESQ")
    ax.legend(loc="best")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    plt.close(fig)
    print("Keine PESQ-Spalte in den eingelesenen Runs gefunden.")

In [ ]:
# Optional: zusammengeführte Tabelle exportieren
export_rows = []
for run_name, df in run_data.items():
    temp = df.copy()
    if "/" in run_name:
        ablation, seed = run_name.split("/", 1)
    else:
        ablation, seed = run_name, ""
    temp.insert(0, "seed", seed)
    temp.insert(0, "ablation", ablation)
    temp.insert(0, "run", run_name)
    export_rows.append(temp)

if export_rows:
    merged = pd.concat(export_rows, ignore_index=True)
    out_csv = Path("run_metrics_merged.csv")
    merged.to_csv(out_csv, index=False)
    print(f"Exportiert: {out_csv.resolve()}")
else:
    print("Nichts zu exportieren.")